# 环境配置

## 安装包管理工具

In [1]:
%pip install uv

!uv --version

/github.com/sammyne/Deep-Learning-with-Python-2ed-cn/chapter14/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
uv 0.7.19


## 安装依赖

In [ ]:
!uv pip install datasets==3.6.0 transformers==4.53.1

⠙                                                                                 × No solution found when resolving dependencies:
  ╰─▶ Because there is no version of tokenizer==0.21.3 and you require
      tokenizer==0.21.3, we can conclude that your requirements are
      unsatisfiable.


In [2]:
!uv add tokenizers==0.21.2

Resolved 48 packages in 0.57ms
Audited 42 packages in 0.02ms


### 11.2.3 建立词表索引

In [ ]:
vocabulary = {}
for text in dataset:
  text = standardize(text)
  tokens = tokenize(text)
  for token in tokens:
    if token not in vocabulary:
      vocabulary[token] = len(vocabulary)

### 11.2.4 使用 TextVectorization 层

In [6]:
import string

class Vectorizer:
    def standardize(self, text):
        text = text.lower()
        return "".join(char for char in text if char not in string.punctuation)

    def tokenize(self, text):
        text = self.standardize(text)
        return text.split()

    def make_vocabulary(self, dataset):
        self.vocabulary = {"": 0, "[UNK]": 1}
        for text in dataset:
            text = self.standardize(text)
            tokens = self.tokenize(text)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)
        self.inverse_vocabulary = dict(
            (v, k) for k, v in self.vocabulary.items())

    def encode(self, text):
        text = self.standardize(text)
        tokens = self.tokenize(text)
        return [self.vocabulary.get(token, 1) for token in tokens]

    def decode(self, int_sequence):
        return " ".join(
            self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

vectorizer = Vectorizer()
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]
vectorizer.make_vocabulary(dataset)

In [7]:
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = vectorizer.encode(test_sentence)
print(encoded_sentence)

[2, 3, 5, 7, 1, 5, 6]


In [8]:
decoded_sentence = vectorizer.decode(encoded_sentence)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


In [3]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import PreTokenizer

class MiniPreTokenizer(PreTokenizer):
  def __init__(self):
    pass

  def pre_tokenize_str(self, sequence):
    return [self.tokenize(v) for v in sequence]

  def tokenize(self, text):
    return "".join(filter(str.isalnum, text.lower())).split()

# 加载预训练的分词器
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = MiniPreTokenizer()

dataset = [
  "I write, erase, rewrite",
  "Erase again, and then",
  "A poppy blooms.",
]

tokenizer.train_from_iterator(dataset, length=len(dataset))

print(tokenizer.get_vocab())
print(tokenizer.get_vocab_size())

TypeError: No constructor defined for MiniPreTokenizer